# **Objective 3:  Identify demographic differences in team merchandise preferences**


Connect to EC2 Instance and F1 Database

In [ ]:
!pip install pymysql

In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import pandas as pd

In [ ]:
# Connect to EC2 instance

# parameters
host_ip = "3.139.88.161"
id = "test1" # Use your 'test1' username
pw = "Test1234#"  # Use your 'Test1234#' password

# connect to mysql server
url = URL.create(
    drivername="mysql+pymysql",
    host=host_ip,
    port=3306,
    username= id,
    password=pw)

# Use create_engine('mysql server address and login info') to create a database engine to connect to MYSQL server.
sql_engine = create_engine(url)
# use .connect() to open the connection to the server for the sql_engine.
sql_connection = sql_engine.connect()

In [ ]:
# Create a connection to the f1_merch database
db_url = URL.create(
    drivername="mysql+pymysql",
    host=host_ip,
    port=3306,
    username=id,
    password=pw,
    database="f1_merch"
)

db_engine = create_engine(db_url)
db_connection = db_engine.connect()

Load tables and only keep the columns needed

In [ ]:
# ELT: load raw data first, transform inside SQL and read it back to Python

db_connection.execute(text("""
DROP TABLE IF EXISTS user_cleaned
"""))

db_connection.execute(text("""
CREATE TABLE user_cleaned AS
SELECT
    user_id,
    TRIM(UPPER(gender)) AS gender,
    age,
    zip_code
FROM user
WHERE user_id IS NOT NULL
"""))

db_connection.commit()

In [ ]:
# load tables needed for objective 3
df_user = pd.read_sql("SELECT * FROM user_cleaned", db_engine)
df_orders = pd.read_sql("SELECT * FROM orders", db_engine)
df_order_items = pd.read_sql("SELECT * FROM order_items", db_engine)
df_product = pd.read_sql("SELECT * FROM product", db_engine)
df_team = pd.read_sql("SELECT * FROM team", db_engine)

print(df_user.head())
print(df_orders.head())
print(df_order_items.head())
print(df_product.head())
print(df_team.head())

In [ ]:
# identify columns needed for objective 3

df_user = df_user[["user_id","age","gender","zip_code"]]
df_orders = df_orders[["order_id","user_id"]]
df_order_items = df_order_items[["order_id","product_id"]]
df_product = df_product[["product_id","team_id","category"]]
df_team = df_team[["team_id","team_name"]]

Preprocessing Step - Cleaning up the data

In [ ]:
# drop nulls for critical columns only

df_user = df_user.dropna(subset=["user_id", "age", "gender"])
df_orders = df_orders.dropna(subset=["order_id", "user_id"])
df_order_items = df_order_items.dropna(subset=["order_id", "product_id"])
df_product = df_product.dropna(subset=["product_id", "team_id", "category"])
df_team = df_team.dropna(subset=["team_id", "team_name"])

In [ ]:
# Ensure the datatype is an integer for certain columns

df_user['user_id'] =df_user['user_id'].astype(int)
df_user['age'] = df_user['age'].astype(int)
df_user['zip_code'] = df_user['zip_code'].astype(int)
df_orders['order_id'] = df_orders['order_id'].astype(int)
df_orders['user_id'] = df_orders['user_id'].astype(int)
df_product['product_id'] = df_product['product_id'].astype(int)
df_order_items['order_id'] = df_order_items['order_id'].astype(int)
df_order_items['product_id'] = df_order_items['product_id'].astype(int)
df_team['team_id'] = df_team['team_id'].astype(int)

In [ ]:
# standardize the product and team_name columns

# turns all values into text, removes extra spaces, and converts into title case
df_product["category"] =df_product["category"].astype(str).str.strip().str.title()
df_team["team_name"]= df_team["team_name"].astype(str).str.strip().str.title()

In [ ]:
valid_categories = ['Top', 'Bottom', 'Outerwear', 'Activewear', 'Accessory']

def clean_category(cat):
    cat = str(cat).strip().lower()

    if cat in ['top', 'tops', 't-shirt', 'shirt', 'tee']:
        return 'Top'
    elif cat in ['bottom', 'bottoms', 'pants', 'shorts']:
        return 'Bottom'
    elif cat in ['outerwear', 'jacket', 'hoodie', 'coat']:
        return 'Outerwear'
    elif cat in ['activewear', 'gym wear', 'sportswear']:
        return 'Activewear'
    elif cat in ['accessory', 'accessories', 'hat', 'cap']:
        return 'Accessory'
    else:
        return 'NA'

df_product['category'] = df_product['category'].apply(clean_category)

In [ ]:
allowed_teams = [
    'Ferrari', 'Redbull', 'Mercedes', 'Mclaren', 'Aston Martin',
    'Alpine', 'Williams', 'Haas', 'Racing Bulls', 'Audi', 'Cadillac'
]

df_team = df_team[df_team["team_name"].isin(allowed_teams)]

df_team.head()

Create demographic segments




In [ ]:
# age groups
def age_group(age):
    if age < 18:
        return "Under 18"
    elif age <= 24:
        return "18 to 24"
    elif age <= 34:
        return "25 to 34"
    elif age <= 44:
        return "35 to 44"
    elif age <= 54:
        return "45 to 54"
    else:
        return "55 Plus"

df_user["age_group"] = df_user["age"].apply(age_group)

In [ ]:
# regions - determined by using the census regions and starting zip codes
def get_region(zip_code):
    zip_code = str(zip_code)
    if zip_code.startswith(("0", "1")):
        return "Northeast"
    elif zip_code.startswith(("2","3")):
        return "Southeast"
    elif zip_code.startswith(("4","5","6")):
        return "Midwest"
    elif zip_code.startswith(("7","85","86","870","884")):
        return "South"
    else:
        return "West"

df_user["region"] = df_user["zip_code"].apply(get_region)

In [ ]:
# combine gender and age group for a gender + age demographic
# will be used for main visuals for clearer patterns
df_user["demo_gender_age"] = (
    df_user["gender"] + " | " + df_user["age_group"]
)

In [ ]:
# combine gender, age_group, and region for a comprehensive demographic segment
# can be used for a deeper analysis
df_user["demographic_segment"] = (
    df_user["gender"] + " | " +
    df_user["age_group"] + " | " +
    df_user["region"]
)

Transformation - Merge Tables

In [ ]:
df_merged = df_orders.merge(df_user, on="user_id", how="inner")
df_merged = df_merged.merge(df_order_items, on="order_id", how="inner")
df_merged = df_merged.merge(df_product, on="product_id", how="inner")
df_merged = df_merged.merge(df_team, on="team_id", how="inner")

print(df_merged.head())
print(df_merged.columns)

Metric 1: Distribution of purchases by team within each demographic segment

In [ ]:
# counts how many purchases each team has within each demographic segment
team_demo_counts = (
    df_merged
    .groupby(["demo_gender_age", "team_name"])
    .size()
    .reset_index(name="purchase_count")
)


print(team_demo_counts.head(50))

In [ ]:
# convert into a percentage

team_demo_counts["segment_total"] = (
    team_demo_counts.groupby("demo_gender_age")["purchase_count"].transform("sum")
)

team_demo_counts["purchase_pct"] = (
    team_demo_counts["purchase_count"] / team_demo_counts["segment_total"] * 100
)

In [ ]:
# identify the dominant team in each demographic

top_team_by_demo = (
    team_demo_counts
    .sort_values(["demo_gender_age", "purchase_count"], ascending=[True, False])
    .groupby("demo_gender_age")
    .head(1)
    .reset_index(drop=True)
)


print(top_team_by_demo)

Metric 2: Percentage of team purchases within each demographic and category segment

In [ ]:
# counts how many purchases each team has within each demographic category

demo_category_team_counts = (
    df_merged
    .groupby(["demo_gender_age", "category", "team_name"])
    .size()
    .reset_index(name="purchase_count")
)

print(demo_category_team_counts.head(20))

In [ ]:
# convert into a percentage

demo_category_team_counts["demo_category_total"] = (
    demo_category_team_counts
    .groupby(["demo_gender_age", "category"])["purchase_count"]
    .transform("sum")
)

demo_category_team_counts["purchase_pct_within_demo_category"] = (
    demo_category_team_counts["purchase_count"] / demo_category_team_counts["demo_category_total"] * 100
).round(2)

print(demo_category_team_counts.head(20))

In [ ]:
# identify dominant team in each demographic and category combination

top_team_by_demo_category = (
    demo_category_team_counts
    .sort_values(
        ["demo_gender_age", "category", "purchase_count"],
        ascending=[True, True, False]
    )
    .groupby(["demo_gender_age", "category"])
    .head(1)
    .reset_index(drop=True)
)

print(top_team_by_demo_category)

Clustering

# Visualizations

In [ ]:
# dictionary mapping each team to its color
# stored in lowercase to match the normalized team names

import re

team_colors = {
    "mercedes": "#27F4D2",
    "red bull racing": "#3671C6",
    "ferrari": "#E8002D",
    "mclaren": "#FF8000",
    "alpine": "#00A1E8",
    "racing bulls": "#6692FF",
    "aston martin": "#229971",
    "williams": "#1868DB",
    "audi": "#FF2D00",
    "haas": "#DEE1E2",
    "cadillac": "#AAAAAD",
}

# standardizes team names. removes extra spaces and converts into lowercase
def normalize_team_name(name):
    name = str(name).strip().lower()
    name = re.sub(r"\s+", " ", name)
    return name

# returns the corresponding color for each time
# if team is not found, defaults to gray
def team_color(name):
    return team_colors.get(normalize_team_name(name), "#888888")


Metric 1 Visual: Distribution of purchases by team within each demographic segment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# shows how purchase preferences vary across demographic groups by comparing
# the share of purchases for each team.
def plot_team_share_by_demo(team_demo_counts, demographic_col="demo_gender_age"):
    pivot = (
        team_demo_counts
        .pivot_table(index=demographic_col, columns="team_name",
                     values="purchase_pct", aggfunc="sum", fill_value=0)
    )

    teams  = pivot.columns.tolist()
    colors = [team_color(t) for t in teams]
    demos  = pivot.index.tolist()

    fig, ax = plt.subplots(figsize=(14, 5))


    bottoms = np.zeros(len(demos))
    for team, color in zip(teams, colors):
        vals = pivot[team].values
        ax.bar(demos, vals, bottom=bottoms, color=color, width=0.7, label=team)
        bottoms += vals

    ax.set_ylim(0, 105)
    ax.set_ylabel("Share of purchases (%)", fontsize=10)
    ax.set_xlabel("")
    ax.set_title("Team purchase share within each demographic segment", fontsize=13, fontweight="bold", pad=12)
    ax.set_xticks(range(len(demos)))
    ax.set_xticklabels(demos, rotation=40, ha="right", fontsize=9)
    ax.legend(bbox_to_anchor=(1.01, 1),loc="upper left",frameon=False, title="Team")


    plt.tight_layout()
    plt.show()


In [ ]:
 # identifies which teams dominate specific age and gender segments

 plot_team_share_by_demo(team_demo_counts)

Metric 2 Visual: Identify the dominant team in each demographic category combination

In [ ]:
# the chart will break down team preferences within product categories
# will give more detailed insights into how demographics influence purchasing bahavior

import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np

def plot_team_cat(df, selected_category, demo_col="demo_gender_age"):

    # filter by selected category
    filtered = df[df["category"] == selected_category]

    # pivot
    pivot = filtered.pivot_table(
        index=demo_col,
        columns="team_name",
        values="purchase_pct_within_demo_category",
        aggfunc="sum",
        fill_value=0
    )

    demos = pivot.index.tolist()
    teams = pivot.columns.tolist()

    x = np.arange(len(demos))
    width = 0.8 / len(teams)

    fig, ax = plt.subplots(figsize=(14, 5))
    fig.patch.set_facecolor("#FAFAFA")

    # grouped bars
    for i, team in enumerate(teams):
        ax.bar(
            x + i * width - (len(teams)-1)*width/2,
            pivot[team].values,
            width=width,
            label=team
        )

    ax.set_title(f"Team Share by Demographic — {selected_category}", fontsize=13, fontweight="bold")
    ax.set_ylabel("Purchase Share (%)")
    ax.set_xticks(x)
    ax.set_xticklabels(demos, rotation=35, ha="right")

    ax.legend(bbox_to_anchor=(1.01,1), loc="upper left", frameon=False, title="Team",title_fontsize = 9, fontsize= 8)
    ax.set_facecolor("#FFFFFF")


    plt.tight_layout()
    plt.show()

In [ ]:
# highlights how team popularity can change depending on both demographic
# and product category

from ipywidgets import interact

# drop down button for categories

def plot_cat_type(category):
    plot_team_cat(demo_category_team_counts, category)

interact(
    plot_cat_type,
    category=sorted(demo_category_team_counts["category"].dropna().unique())
)

In [ ]:
# extra heatmap using full demographic segment (age,gender,region)
# purchase counts for teams within each demographic segment
import seaborn as sns
import matplotlib.pyplot as plt

# create pivot: segment and team
pivot = (
    df_merged
    .groupby(["demographic_segment", "team_name"])
    .size()
    .reset_index(name="count")
    .pivot(index="demographic_segment", columns="team_name", values="count")
    .fillna(0)
)

plt.figure(figsize=(12, 8))
sns.heatmap(pivot, cmap="coolwarm", linewidths=0.5)

plt.title("Team Preference by Full Demographic Segment", fontsize=14, fontweight="bold")
plt.xlabel("Team")
plt.ylabel("Demographic Segment")

plt.tight_layout()
plt.show()

For reverse ETL: save results back to the database

In [ ]:
team_demo_counts.to_sql("team_demo_distribution", db_engine, if_exists="replace", index=False)
top_team_by_demo.to_sql("top_team_by_demographic", db_engine, if_exists="replace", index=False)

demo_category_team_counts.to_sql("demo_category_team_distribution", db_engine, if_exists="replace", index=False)
top_team_by_demo_category.to_sql("top_team_by_demo_category", db_engine, if_exists="replace", index=False)

In [ ]:
# close the connection
sql_connection.close()
db_engine.dispose()

print ("Connection closed.")